# OP-Cal — Notebook 08 : sensibilité aux petits hôpitaux

**Analyse exploratoire sur les 60 hôpitaux de développement uniquement.**  
Le notebook sous-échantillonne les observations de développement, recalcule les descripteurs label-free, mesure la stabilité de la pente `b` et les métriques de calibration. Il ne lit aucun label du test verrouillé, ne modifie pas le protocole confirmatoire et ne doit pas servir à choisir une méthode sur le test verrouillé.

Les simulations sont exploratoires. La procédure de correction de taille éventuelle n'est pas confirmatoire.

In [1]:
# 0. Imports, chemins et garde-fous
import sys, json, hashlib, warnings, importlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit, logit

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 60)

PROJECT_DIR = Path("..")
DATA_DIR = PROJECT_DIR / "data_v1"
SRC_DIR = PROJECT_DIR / "src"
RES02 = PROJECT_DIR / "results" / "02_baselines"
RES04 = PROJECT_DIR / "results" / "04_observation_label"
RES06B = PROJECT_DIR / "results" / "06b_faisabilite_opcal"
RES06C = PROJECT_DIR / "results" / "06c_robustesse_opcal"
RES07 = PROJECT_DIR / "results" / "07_confirmation"
OUT_DIR = PROJECT_DIR / "results" / "08_small_sites_dev"
OUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC_DIR.resolve()))

import opcal_feas as oc
import opcal_amend as am
importlib.reload(oc)
importlib.reload(am)

protocol = json.loads((DATA_DIR / "splits" / "protocol.json").read_text())
assert protocol["hash"] == "d92099ea7c11d517", "Protocole inattendu."
FOLDS = sorted(int(k) for k in protocol["folds"])
DEV = sorted(int(h) for k in protocol["folds"] for h in protocol["folds"][k])
LOCKED = set(int(h) for h in protocol["locked_test_hospitals"])
assert len(DEV) == 60 and len(LOCKED) == 96 and not (set(DEV) & LOCKED)

SPEC_PATH = RES06C / "specification_gelee.json"
SPEC = json.loads(SPEC_PATH.read_text())
FROZEN = list(SPEC["meta_modele"]["descripteurs"])
assert len(FROZEN) == 44

PRED_DIR = RES02 / "preds"
OBS_DEV_PATH = RES07 / "observation_developpement.parquet"
D_SITE_PATH = RES06B / "descripteurs_site.csv"
assert OBS_DEV_PATH.exists(), f"Fichier absent : {OBS_DEV_PATH}"
assert D_SITE_PATH.exists(), f"Fichier absent : {D_SITE_PATH}"

SEED = 20261029
N_REP = 200
TARGET_TOTAL_SIZES = [100, 150, 250, 350, 499]
MIN_EVAL_N = 20
print(f"DEV={len(DEV)} hôpitaux; LOCKED labels non utilisés; repetitions={N_REP}")

DEV=60 hôpitaux; LOCKED labels non utilisés; repetitions=200


In [2]:
# 1. Charger les seules données de développement
PRED, _ = oc.load_transfer(PRED_DIR, "x_only", FOLDS)
PRED = PRED[PRED["hospitalid"].isin(DEV)].copy()
assert not (set(PRED["hospitalid"].unique()) & LOCKED)
assert {"hospitalid", "stay_id", "y", "p"} <= set(PRED.columns)

D_SITE = pd.read_csv(D_SITE_PATH, index_col=0)
D_SITE.index = D_SITE.index.astype(int)
D_SITE = D_SITE.loc[DEV].copy()
OBS = pd.read_parquet(OBS_DEV_PATH)
assert set(OBS["hospitalid"].unique()) <= set(DEV), "OBS contient des hôpitaux hors DEV."
assert "split" in OBS.columns
assert set(FROZEN) <= set(D_SITE.columns)

print("Prédictions OOF:", PRED.shape, "| descripteurs:", D_SITE.shape, "| observations:", OBS.shape)
display(PRED.groupby("hospitalid").size().describe().to_frame("n_predictions"))

AssertionError: 

## Principe

Pour chaque répétition :
1. échantillonner les séjours `train` de chaque hôpital pour simuler un site plus petit ;
2. recalculer les descripteurs avec `oc.site_descriptors` ;
3. échantillonner les prédictions cross-fittées de développement pour simuler une estimation du CITL sur peu de patients ;
4. ré-estimer OP-Cal en laissant un fold entier hors du méta-modèle ;
5. mesurer la pente, le |CITL| avant/après correction et le gain de log-vraisemblance vs E0.

Le code utilise uniquement les labels des 60 hôpitaux de développement.

In [ ]:
# 2. Fonctions de simulation
def sample_by_hospital(df, n, rng, replace=False):
    parts = []
    for h, g in df.groupby("hospitalid", sort=False):
        if len(g) == 0:
            continue
        nn = n if replace else min(n, len(g))
        parts.append(g.sample(n=nn, replace=replace,
                              random_state=int(rng.integers(0, 2**31-1))))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0].copy()


def sampled_descriptors(target_total_n, rng, rep):
    train = OBS[OBS["split"].astype(str).str.lower().eq("train")].copy()
    n_train = max(10, int(round(0.8 * target_total_n)))
    obs_sample = sample_by_hospital(train, n_train, rng, replace=False)
    tmp = OUT_DIR / f"_obs_tmp_{target_total_n}_{rep}.parquet"
    obs_sample.to_parquet(tmp, index=False)
    try:
        d, _ = oc.site_descriptors(DATA_DIR, tmp, DEV)
    finally:
        if tmp.exists():
            tmp.unlink()
    d.index = d.index.astype(int)
    # Certaines colonnes unit_* peuvent être omises si un type d'unité n'est pas
    # présent dans l'échantillon. Arrêt explicite plutôt que d'inventer une valeur.
    missing = set(FROZEN) - set(d.columns)
    if missing:
        raise ValueError(
            f"Descripteurs gelés manquants après recalcul : {sorted(missing)}. "
            "Adapter le calcul des parts unit_* en reprenant opcal_final.unit_shares."
        )
    return d.reindex(DEV)[FROZEN]


def sampled_predictions(target_n, rng):
    # La taille d'évaluation cible correspond à environ 20 % de la taille totale.
    n_eval = max(MIN_EVAL_N, int(round(0.20 * target_n)))
    parts = []
    for h, g in PRED.groupby("hospitalid", sort=False):
        # Bootstrap avec remise si les prédictions OOF disponibles sont insuffisantes.
        repl = len(g) < n_eval
        ss = g.sample(n=n_eval if repl else min(n_eval, len(g)),
                      replace=repl, random_state=int(rng.integers(0, 2**31-1))).copy()
        ss["stay_id"] = [f"sim{h}_{i}" for i in range(len(ss))]
        parts.append(ss)
    return pd.concat(parts, ignore_index=True)


def crossfit_opcal(cal, desc):
    out = pd.Series(index=cal.index, dtype=float, name="a_hat")
    for fold in FOLDS:
        fold_h = [int(h) for h in protocol["folds"][str(fold)]]
        test_h = [h for h in fold_h if h in cal.index and h in desc.index]
        train_h = [h for h in DEV if h not in test_h and h in cal.index and h in desc.index]
        if len(test_h) == 0 or len(train_h) < 10:
            continue
        model = oc.SiteEB().fit(
            desc.loc[train_h, FROZEN].to_numpy(float),
            cal.loc[train_h, "citl"].to_numpy(float),
            cal.loc[train_h, "var_citl"].to_numpy(float))
        mean, _ = model.predict(desc.loc[test_h, FROZEN].to_numpy(float))
        out.loc[test_h] = np.asarray(mean, dtype=float)
    return out


def calculate_metrics(cal, ahat, patient_df):
    common = cal.index.intersection(ahat.dropna().index)
    cal, ahat = cal.loc[common].copy(), ahat.loc[common]
    res = {"n_hospitals": len(common)}
    try:
        if len(common) >= 10:
            c1 = am.c1_analysis(cal, ahat - ahat.mean())
            res.update({"b": float(c1["b"]), "b_ci_low": float(c1["b_IC_bas"]),
                        "b_ci_high": float(c1["b_IC_haut"]),
                        "p_b_gt_0": float(c1["p_C1a"]),
                        "p_b_gt_0_5": float(c1["p_C1b"])})
    except Exception as e:
        res["slope_error"] = str(e)

    res["mean_abs_citl_uncorrected"] = float(cal["citl"].abs().mean())
    res["mean_abs_citl_corrected"] = float((cal["citl"] - ahat).abs().mean())
    res["median_abs_citl_uncorrected"] = float(cal["citl"].abs().median())
    res["median_abs_citl_corrected"] = float((cal["citl"] - ahat).abs().median())

    pdat = patient_df[patient_df["hospitalid"].isin(common)].copy()
    pdat = pdat[pdat["y"].notna() & pdat["p"].notna()]
    if len(pdat):
        pdat["a_hat"] = pdat["hospitalid"].map(ahat)
        p0 = np.clip(pdat["p"].to_numpy(float), 1e-6, 1-1e-6)
        p1 = expit(logit(p0) + pdat["a_hat"].to_numpy(float))
        y = pdat["y"].to_numpy(float)
        ll0 = y*np.log(p0) + (1-y)*np.log1p(-p0)
        ll1 = y*np.log(p1) + (1-y)*np.log1p(-p1)
        res["ll_gain_per_patient_vs_E0"] = float(np.mean(ll1-ll0))
        res["n_eval_rows"] = int(len(pdat))
    return res

In [ ]:
# 3. Smoke test avant de lancer toutes les répétitions
rng = np.random.default_rng(SEED)
D_SMOKE = sampled_descriptors(TARGET_TOTAL_SIZES[0], rng, rep=0)
P_SMOKE = sampled_predictions(TARGET_TOTAL_SIZES[0], rng)
CAL_SMOKE = oc.hospital_calibration(P_SMOKE)
print("Descripteurs recalculés:", D_SMOKE.shape)
print("Calibration simulée:", CAL_SMOKE.shape)
display(CAL_SMOKE.head())

In [ ]:
# 4. Simulation principale
rows = []
for target_n in TARGET_TOTAL_SIZES:
    print(f"Taille cible {target_n}")
    for rep in range(N_REP):
        rng = np.random.default_rng(SEED + target_n*100000 + rep)
        D_SIM = sampled_descriptors(target_n, rng, rep)
        P_SIM = sampled_predictions(target_n, rng)
        CAL_SIM = oc.hospital_calibration(P_SIM)
        CAL_SIM = CAL_SIM.loc[CAL_SIM.index.isin(DEV)].copy()
        D_SIM = D_SIM.loc[D_SIM.index.isin(CAL_SIM.index)].copy()
        A_SIM = crossfit_opcal(CAL_SIM, D_SIM)
        row = calculate_metrics(CAL_SIM, A_SIM, P_SIM)
        row.update({"target_total_n": target_n, "rep": rep,
                    "seed": SEED + target_n*100000 + rep})
        rows.append(row)

SIM = pd.DataFrame(rows)
assert len(SIM), "Aucun résultat : vérifier les APIs et les chemins."
SIM.to_csv(OUT_DIR / "simulation_small_sites_repetitions.csv", index=False)
display(SIM.head())
print("Simulations terminées:", len(SIM))

In [ ]:
# 5. Résumé par taille cible
metrics = ["n_hospitals", "b", "mean_abs_citl_uncorrected",
           "mean_abs_citl_corrected", "ll_gain_per_patient_vs_E0",
           "p_b_gt_0", "p_b_gt_0_5"]
summary_rows = []
for n, g in SIM.groupby("target_total_n", sort=True):
    row = {"target_total_n": n, "n_repetitions": len(g)}
    for m in metrics:
        if m in g.columns:
            v = pd.to_numeric(g[m], errors="coerce").dropna()
            row[m+"_median"] = v.median() if len(v) else np.nan
            row[m+"_q025"] = v.quantile(.025) if len(v) else np.nan
            row[m+"_q975"] = v.quantile(.975) if len(v) else np.nan
    summary_rows.append(row)
SUMMARY = pd.DataFrame(summary_rows).set_index("target_total_n")
SUMMARY.to_csv(OUT_DIR / "summary_small_sites.csv")
display(SUMMARY.round(4))

In [ ]:
# 6. Figures descriptives
for metric, ylabel, title in [
    ("b", "Pente b", "Stabilité de la pente"),
    ("mean_abs_citl_uncorrected", "|CITL| moyen", "Calibration sans correction"),
    ("mean_abs_citl_corrected", "|CITL| moyen", "Calibration après OP-Cal"),
    ("ll_gain_per_patient_vs_E0", "Gain log-vraisemblance/patient", "Gain vs absence de correction"),
]:
    if metric not in SIM.columns:
        continue
    g = SIM.groupby("target_total_n")[metric].agg(
        median="median", low=lambda x: x.quantile(.025), high=lambda x: x.quantile(.975)).dropna()
    if g.empty:
        continue
    x = g.index.to_numpy(float)
    plt.figure(figsize=(6.5, 4.2))
    plt.plot(x, g["median"].to_numpy(float), marker="o")
    plt.fill_between(x, g["low"].to_numpy(float), g["high"].to_numpy(float), alpha=.2)
    plt.xlabel("Taille totale simulée par hôpital")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=.25)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{metric}.png", dpi=180)
    plt.show()

In [ ]:
# 7. Rapport d'exécution et empreintes
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

summary_path = OUT_DIR / "summary_small_sites.csv"
raw_path = OUT_DIR / "simulation_small_sites_repetitions.csv"
report = {
    "analysis": "OP-Cal notebook 08 — exploratory small-site sensitivity",
    "status": "exploratory_development_only",
    "protocol_hash": protocol["hash"],
    "specification_sha256": sha256_file(SPEC_PATH),
    "development_hospitals_n": len(DEV),
    "locked_hospital_labels_used": False,
    "seed": SEED,
    "n_repetitions_per_size": N_REP,
    "target_total_sizes": TARGET_TOTAL_SIZES,
    "n_simulation_rows": int(len(SIM)),
    "summary_sha256": sha256_file(summary_path),
    "repetitions_sha256": sha256_file(raw_path),
    "warning": "Exploratory only; bootstrap may duplicate development patients."
}
(OUT_DIR / "rapport_08.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print(json.dumps(report, indent=2))